# MRIxFields2026 Task 3 — StarGAN v2 training on Colab

This notebook trains the organizers' single shared 15-domain Task 3 model. Complete and report the pretrained submission milestone first. Every training switch defaults to off.

Put `task3_colab_train.py` and `requirements-task3-training-colab.txt` together in your private Drive source folder. Select a GPU runtime and set the dataset root containing `training_prospective`, not a split directory. The staging cell copies code, selected training splits, and an optional initialization checkpoint into the separate `/content/mrixfields_task3/training/` tree, verifies the copies, and unmounts Drive before processing.

Validation inference and submission packaging belong to the separate [pretrained baseline notebook](task3_pretrained_baseline_colab.ipynb). Runtime storage disappears when Colab resets; export one selected checkpoint and its provenance after a successful run.


In [ ]:
from google.colab import drive

from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

DRIVE_MOUNT = Path('/content/drive')
RUNTIME_ROOT_VALUE = '/content/mrixfields_task3/training'  # @param {type:'string'}
DRIVE_CODE_SOURCE_VALUE = '/content/drive/MyDrive/mrixfields_task3_source'  # @param {type:'string'}
DRIVE_DATA_SOURCE_VALUE = '/content/drive/MyDrive/mrixfields2026'  # @param {type:'string'}
DRIVE_CHECKPOINT_SOURCE_VALUE = '/content/drive/MyDrive/mrixfields_weights/task3_any_to_any_multimodal/stargan_v2/pro_pretrained/weights/checkpoint_epoch50.pth'  # @param {type:'string'}
DRIVE_EXPORT_ROOT_VALUE = '/content/drive/MyDrive/mrixfields_task3_exports/training'  # @param {type:'string'}

RUNTIME_ROOT = Path(RUNTIME_ROOT_VALUE).expanduser().resolve()
DRIVE_CODE_SOURCE = Path(DRIVE_CODE_SOURCE_VALUE)
DRIVE_DATA_SOURCE = Path(DRIVE_DATA_SOURCE_VALUE)
DRIVE_CHECKPOINT_SOURCE = Path(DRIVE_CHECKPOINT_SOURCE_VALUE)
DRIVE_EXPORT_ROOT = Path(DRIVE_EXPORT_ROOT_VALUE)
drive_root = DRIVE_MOUNT.resolve()
if Path('/content').resolve() not in RUNTIME_ROOT.parents:
    raise ValueError('Choose a working directory below /content.')

CODE_DIR = RUNTIME_ROOT / 'code'
DATA_DIR = RUNTIME_ROOT / 'data'
DATA_VIEW_DIR = RUNTIME_ROOT / 'data_view'
PREPROCESSED_DIR = RUNTIME_ROOT / 'preprocessed'
WEIGHTS_DIR = RUNTIME_ROOT / 'weights'
RESULTS_DIR = RUNTIME_ROOT / 'results'
OUTPUT_DIR = RESULTS_DIR / 'training_runs'
REPORT_DIR = RESULTS_DIR / 'reports'
OFFICIAL_REPO = RUNTIME_ROOT / 'official' / 'MRIxFields2026'
HELPER = CODE_DIR / 'task3_colab_train.py'
REQUIREMENTS = CODE_DIR / 'requirements-task3-training-colab.txt'
RUNTIME_WORK_PATHS = (
    CODE_DIR, DATA_DIR, DATA_VIEW_DIR, PREPROCESSED_DIR,
    WEIGHTS_DIR, RESULTS_DIR, OUTPUT_DIR, REPORT_DIR, OFFICIAL_REPO,
)


def require_runtime_path(path):
    path = Path(path).expanduser().resolve()
    if path == drive_root or drive_root in path.parents:
        raise ValueError(f'Working path points to mounted Drive: {path}')
    if path != RUNTIME_ROOT and RUNTIME_ROOT not in path.parents:
        raise ValueError(f'Working path must stay inside {RUNTIME_ROOT}: {path}')
    return path


def drive_is_mounted():
    return os.path.ismount(DRIVE_MOUNT) or (DRIVE_MOUNT / 'MyDrive').is_dir()


def assert_runtime_only():
    if drive_is_mounted():
        raise RuntimeError('Finish input staging and unmount Drive before processing.')
    for path in RUNTIME_WORK_PATHS:
        require_runtime_path(path)


def sha256_path(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def copy_verified_file(source, destination):
    source = Path(source).expanduser().resolve()
    destination = Path(destination).expanduser().resolve()
    if not source.is_file():
        raise FileNotFoundError(source)
    if source == destination:
        raise ValueError('Copy source and destination must be different files.')
    size = source.stat().st_size
    digest = sha256_path(source)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f'.{destination.name}.part')
    shutil.copy2(source, temporary)
    if temporary.stat().st_size != size or sha256_path(temporary) != digest:
        raise RuntimeError(f'Copy verification failed: {destination}')
    os.replace(temporary, destination)
    return {'name': destination.name, 'bytes': size, 'sha256': digest}


for path in RUNTIME_WORK_PATHS:
    require_runtime_path(path)
for path in (CODE_DIR, WEIGHTS_DIR, REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(require_runtime_path(RUNTIME_ROOT / '.cache' / 'torch'))
os.environ['HF_HOME'] = str(require_runtime_path(RUNTIME_ROOT / '.cache' / 'huggingface'))
drive.mount(str(DRIVE_MOUNT))
print('Training runtime:', RUNTIME_ROOT)


## Stage inputs and close the Drive boundary

The dataset path must be the parent root containing the prospective training split. Organizer and lowercase split names are resolved by the existing training helper. Prospective staging is on; retrospective data and official initialization are optional and off by default. Select everything needed now, because processing cells will require Drive to be unmounted.


In [ ]:
STAGE_PROSPECTIVE_DATA = True  # @param {type:'boolean'}
STAGE_RETROSPECTIVE_DATA = False  # @param {type:'boolean'}
USE_EXISTING_CHECKPOINT = False  # @param {type:'boolean'}
DOWNLOAD_OFFICIAL_CHECKPOINT = False  # @param {type:'boolean'}
HF_REVISION = 'f1a7d6b741b64c81871d0856fa2d7295d564c141'
HF_FILE = 'task3_any_to_any_multimodal/stargan_v2/pro_pretrained/weights/checkpoint_epoch50.pth'
EXPECTED_CHECKPOINT_SIZE = 631346455
EXPECTED_CHECKPOINT_SHA256 = 'c2f8172c1fbdbf0426e22c7d3d2014632f175a38cf818f95e72b1917badabdeb'
OFFICIAL_CHECKPOINT = require_runtime_path(WEIGHTS_DIR / HF_FILE)


def stage_nifti_tree(source, destination):
    source = Path(source).expanduser().resolve()
    destination = require_runtime_path(destination)
    if not source.is_dir():
        raise FileNotFoundError(source)
    if source == destination or source in destination.parents or destination in source.parents:
        raise ValueError('Source and runtime destination must be separate trees.')
    source_index = {
        path.relative_to(source): path.stat().st_size
        for path in sorted(source.rglob('*.nii.gz'))
    }
    if not source_index:
        raise RuntimeError(f'No NIfTI files found in {source}')
    destination.mkdir(parents=True, exist_ok=True)
    for index, (relative, size) in enumerate(source_index.items(), start=1):
        target = require_runtime_path(destination / relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        temporary = target.with_name(f'.{target.name}.part')
        shutil.copy2(source / relative, temporary)
        if temporary.stat().st_size != size:
            raise RuntimeError(f'Incomplete runtime copy: {temporary}')
        os.replace(temporary, target)
        if index % 10 == 0 or index == len(source_index):
            print(f'Staged {index}/{len(source_index)} NIfTI files')
    runtime_index = {
        path.relative_to(destination): path.stat().st_size
        for path in destination.rglob('*.nii.gz')
    }
    if runtime_index != source_index:
        raise RuntimeError('Runtime NIfTI inventory differs from the selected source tree.')
    return destination


code_files = []
for filename in ('task3_colab_train.py', 'requirements-task3-training-colab.txt'):
    target = require_runtime_path(CODE_DIR / filename)
    source_path = next(
        (path for path in (DRIVE_CODE_SOURCE / filename, Path('/content') / filename)
         if path.is_file()), None,
    )
    if source_path is None:
        raise FileNotFoundError(f'Put {filename} in {DRIVE_CODE_SOURCE} or /content.')
    code_files.append(copy_verified_file(source_path, target))

sys.path.insert(0, str(CODE_DIR))
import task3_colab_train as training_helper

if STAGE_PROSPECTIVE_DATA:
    source_split = training_helper.resolve_split_dir(
        DRIVE_DATA_SOURCE, training_helper.CANONICAL_SPLITS['pro_train']
    )
    stage_nifti_tree(source_split, DATA_DIR / 'training_prospective')
if not any((DATA_DIR / 'training_prospective').rglob('*.nii.gz')):
    raise RuntimeError('Stage prospective training data before closing Drive.')
if STAGE_RETROSPECTIVE_DATA:
    source_split = training_helper.resolve_split_dir(
        DRIVE_DATA_SOURCE, training_helper.CANONICAL_SPLITS['retro_train']
    )
    stage_nifti_tree(source_split, DATA_DIR / 'training_retrospective')

if USE_EXISTING_CHECKPOINT and DOWNLOAD_OFFICIAL_CHECKPOINT:
    raise ValueError('Choose a saved checkpoint or download, not both.')
if USE_EXISTING_CHECKPOINT:
    saved_checkpoint = DRIVE_CHECKPOINT_SOURCE
    if saved_checkpoint.is_dir():
        saved_checkpoint = saved_checkpoint / 'checkpoint_epoch50.pth'
    copy_verified_file(saved_checkpoint, OFFICIAL_CHECKPOINT)
checkpoint_sha256 = None
if OFFICIAL_CHECKPOINT.is_file():
    checkpoint_sha256 = sha256_path(OFFICIAL_CHECKPOINT)
    if (OFFICIAL_CHECKPOINT.stat().st_size != EXPECTED_CHECKPOINT_SIZE
            or checkpoint_sha256 != EXPECTED_CHECKPOINT_SHA256):
        raise RuntimeError('The initialization checkpoint does not match the official size/SHA-256.')
stage_manifest = {
    'source_data_root': str(DRIVE_DATA_SOURCE), 'code_files': code_files,
    'prospective_files': sum(1 for _ in (DATA_DIR / 'training_prospective').rglob('*.nii.gz')),
    'retrospective_staged': STAGE_RETROSPECTIVE_DATA,
    'hf_revision': HF_REVISION, 'checkpoint_sha256': checkpoint_sha256,
}
(REPORT_DIR / 'runtime_staging.json').write_text(
    json.dumps(stage_manifest, indent=2) + '\n', encoding='utf-8'
)
drive.flush_and_unmount()
assert_runtime_only()
print('Runtime staging complete. Drive is unmounted; processing is runtime-only.')


## Install only Task 3 training dependencies

Colab already supplies a matching CUDA PyTorch/torchvision build, so the requirements deliberately do not replace it.

In [ ]:
assert_runtime_only()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQUIREMENTS)])
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU. In Colab choose Runtime > Change runtime type > GPU.')
props = torch.cuda.get_device_properties(0)
print(torch.__version__, torch.version.cuda)
print(props.name, round(props.total_memory / 2**30, 1), 'GiB')

## Fetch the pinned organizer code

This preparation pin matches the training helper. It is not a claim about the current upstream revision. The helper refuses to train with a different checkout, so review any pin update together with the matching Task 3 config and weights.


In [ ]:
assert_runtime_only()
OFFICIAL_COMMIT = 'f0f4ab2c54ab878b48ddfea8d13f4da512ca411e'
if not (OFFICIAL_REPO / '.git').is_dir():
    if OFFICIAL_REPO.exists():
        raise RuntimeError(f'{OFFICIAL_REPO} exists but is not an organizer Git checkout')
    subprocess.check_call(['git', 'clone', 'https://github.com/MRIxFields/MRIxFields2026.git', str(OFFICIAL_REPO)])
subprocess.check_call(['git', '-C', str(OFFICIAL_REPO), 'fetch', 'origin', OFFICIAL_COMMIT])
subprocess.check_call(['git', '-C', str(OFFICIAL_REPO), 'checkout', '--detach', OFFICIAL_COMMIT])
revision = subprocess.check_output(['git', '-C', str(OFFICIAL_REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert revision == OFFICIAL_COMMIT, revision
print('Official revision:', revision)

## Audit prospective training data

This accepts either organizer casing (`Training_prospective`) or the reported local lowercase casing (`training_prospective`). It checks all 15 modality/field groups, filenames, pairing, shapes, dtypes, and voxel sizes without changing the source data. Add `--check-values` only when you want to read every voxel.

In [ ]:
assert_runtime_only()
subprocess.check_call([
    sys.executable, str(HELPER), 'audit-raw',
    '--data-dir', str(DATA_DIR), '--splits', 'pro_train',
    '--report', str(REPORT_DIR / 'pro_train_raw_audit.json'),
])
subprocess.check_call([
    sys.executable, str(HELPER), 'make-data-view',
    '--data-dir', str(DATA_DIR), '--view-dir', str(DATA_VIEW_DIR),
    '--splits', 'pro_train',
])

## Extract the organizer's 220 axial training slices

This is resumable (`--skip_existing`) and writes float32 `[0,1]` NPZ slices to the Colab runtime disk. Prospective extraction is enough for fine-tuning. Retrospective extraction is separate below because it is much larger. The runtime cache is intentionally temporary.

In [ ]:
assert_runtime_only()
RUN_PROSPECTIVE_PREPROCESS = False  # @param {type:'boolean'}
if RUN_PROSPECTIVE_PREPROCESS:
    subprocess.check_call([
        sys.executable, str(HELPER), 'preprocess',
        '--official-repo', str(OFFICIAL_REPO), '--data-dir', str(DATA_DIR),
        '--view-dir', str(DATA_VIEW_DIR), '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--splits', 'pro_train',
    ])
else:
    print('Prospective preprocessing is OFF. Enable it only after the raw audit passes.')

In [ ]:
assert_runtime_only()
RUN_CACHE_AUDIT = False  # @param {type:'boolean'}
if RUN_CACHE_AUDIT:
    subprocess.check_call([
        sys.executable, str(HELPER), 'audit-cache',
        '--preprocessed-dir', str(PREPROCESSED_DIR), '--splits', 'pro_train',
        '--report', str(REPORT_DIR / 'pro_train_cache_audit.json'),
    ])
else:
    print('Cache audit is OFF. Run it after prospective preprocessing.')

## Baseline milestone gate and optional initialization

Leave the acknowledgement false until the official pretrained archive has been built, validated, and reported. The default pilot starts from scratch and needs no initialization checkpoint. For all-subject continuation, select the official checkpoint in the earlier staging cell; it must already be verified in runtime storage before training.


In [ ]:
assert_runtime_only()
BASELINE_MILESTONE_COMPLETE = False  # @param {type:'boolean'}
if DOWNLOAD_OFFICIAL_CHECKPOINT:
    from huggingface_hub import hf_hub_download
    OFFICIAL_CHECKPOINT = require_runtime_path(hf_hub_download(
        repo_id='mrixfields/MRIxFields2026-Baseline', filename=HF_FILE,
        revision=HF_REVISION, local_dir=WEIGHTS_DIR,
    ))
if OFFICIAL_CHECKPOINT.is_file():
    checkpoint_sha256 = sha256_path(OFFICIAL_CHECKPOINT)
    if (OFFICIAL_CHECKPOINT.stat().st_size != EXPECTED_CHECKPOINT_SIZE
            or checkpoint_sha256 != EXPECTED_CHECKPOINT_SHA256):
        raise RuntimeError('The initialization checkpoint does not match the official size/SHA-256.')
    stage_manifest['checkpoint_sha256'] = checkpoint_sha256
    (REPORT_DIR / 'runtime_staging.json').write_text(
        json.dumps(stage_manifest, indent=2) + '\n', encoding='utf-8'
    )
print('Training unlocked:', BASELINE_MILESTONE_COMPLETE)
print('Official initialization staged:', OFFICIAL_CHECKPOINT.is_file())


## Pilot: leakage-free subject-held-out prospective training

Run this first after the milestone. `auto` withholds one complete prospective volunteer at subject level and reports bounded **slice-level proxy** nRMSE/SSIM/LPIPS separately for T1W, T2W, and T2FLAIR. The released `pro_pretrained/epoch50` checkpoint has already seen every prospective training volunteer, so it must not initialize an honest holdout. This default pilot starts from scratch. For a stronger honest pilot, initialize from your own retrospective-only checkpoint and attest that it excludes the holdout. These proxies are not public-validation or leaderboard scores.

In [ ]:
assert_runtime_only()
RUN_PILOT = False  # @param {type:'boolean'}
PILOT_EPOCHS = 3  # @param {type:'integer'}
PILOT_MICROBATCHES = 100  # @param {type:'integer'}
if RUN_PILOT:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Baseline milestone is not acknowledged; training remains locked.')
    subprocess.check_call([
        sys.executable, str(HELPER), 'finetune',
        '--official-repo', str(OFFICIAL_REPO), '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--output-dir', str(OUTPUT_DIR), '--run-name', 'pilot_subject_holdout',
        '--from-scratch', '--holdout-subject', 'auto',
        '--epochs', str(PILOT_EPOCHS), '--max-steps-per-epoch', str(PILOT_MICROBATCHES),
        '--batch-size', '1', '--gradient-accumulation', '1', '--workers', '2',
        '--save-every-steps', '50', '--max-validation-batches', '120',
        '--baseline-milestone-complete',
    ])
else:
    print('Pilot training is OFF.')

### Compare leakage-free pilot epochs by modality

For nRMSE and LPIPS, lower is better; for SSIM, higher is better. These are fixed, balanced slice proxies on the held-out training volunteer, not public-validation scores.

In [ ]:
assert_runtime_only()
import json
import pandas as pd
metric_log = OUTPUT_DIR / 'task3_any_to_any_multimodal' / 'stargan_v2' / 'pilot_subject_holdout' / 'train.jsonl'
rows = []
if metric_log.is_file():
    for line in metric_log.read_text().splitlines():
        record = json.loads(line)
        if 'holdout' not in record:
            continue
        groups = {'ALL': record['holdout']['overall'], **record['holdout']['by_modality']}
        for modality, metrics in groups.items():
            rows.append({
                'epoch': record['epoch'], 'modality': modality,
                'nRMSE': metrics['slice_proxy_nrmse'],
                'SSIM': metrics['slice_proxy_ssim'],
                'LPIPS': metrics['slice_proxy_lpips'],
                'samples': metrics['samples'],
            })
    display(pd.DataFrame(rows).sort_values(['epoch', 'modality']))
else:
    print('Run the leakage-free pilot first:', metric_log)

## Continued fine-tuning of the released baseline on all subjects

This is a different experiment from the leakage-free pilot: it continues the released epoch-50 baseline using all prospective training subjects. It has no honest local holdout, so compare it with the released baseline through a complete public-validation submission, not the pilot proxy. Start conservatively (for example one additional epoch), because the initialization is already fine-tuned. Batch 1 with accumulation 8 matches the organizer config's optimization batch of 8 while fitting smaller GPUs. Checkpoints are atomic, written every 500 microbatches, auto-resumed within an epoch, and only the newest three are retained.

In [ ]:
assert_runtime_only()
RUN_FULL_FINETUNE = False  # @param {type:'boolean'}
FULL_EPOCHS = 1  # @param {type:'integer'}
MAX_RUNTIME_MINUTES = 300  # @param {type:'integer'}
if RUN_FULL_FINETUNE:
    if not OFFICIAL_CHECKPOINT.is_file():
        raise FileNotFoundError('Select an official initialization checkpoint in the staging cell first.')
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Baseline milestone is not acknowledged; training remains locked.')
    subprocess.check_call([
        sys.executable, str(HELPER), 'finetune',
        '--official-repo', str(OFFICIAL_REPO), '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--output-dir', str(OUTPUT_DIR), '--run-name', 'pro_pretrained_all_subjects',
        '--init-checkpoint', str(OFFICIAL_CHECKPOINT), '--holdout-subject', 'none',
        '--epochs', str(FULL_EPOCHS), '--batch-size', '1', '--gradient-accumulation', '8',
        '--workers', '2', '--save-every-steps', '500', '--keep-last', '3',
        '--max-runtime-minutes', str(MAX_RUNTIME_MINUTES),
        '--baseline-milestone-complete',
    ])
else:
    print('Full fine-tuning is OFF.')

## Optional: retrospective pretraining from scratch

Select `STAGE_RETROSPECTIVE_DATA` in the initial staging cell before unmounting Drive. This larger experiment uses the official 200,000-step schedule and remains off unless explicitly selected after the baseline milestone. Its input data and caches stay in the separate training runtime tree.


In [ ]:
assert_runtime_only()
RUN_RETROSPECTIVE_PREPROCESS = False  # @param {type:'boolean'}
if RUN_RETROSPECTIVE_PREPROCESS:
    subprocess.check_call([
        sys.executable, str(HELPER), 'preprocess',
        '--official-repo', str(OFFICIAL_REPO), '--data-dir', str(DATA_DIR),
        '--view-dir', str(DATA_VIEW_DIR), '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--splits', 'retro_train',
    ])
else:
    print('Retrospective preprocessing is OFF.')

In [ ]:
assert_runtime_only()
RUN_RETROSPECTIVE_PRETRAIN = False  # @param {type:'boolean'}
PRETRAIN_STEPS = 200000  # @param {type:'integer'}
if RUN_RETROSPECTIVE_PRETRAIN:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Baseline milestone is not acknowledged; training remains locked.')
    subprocess.check_call([
        sys.executable, str(HELPER), 'pretrain',
        '--official-repo', str(OFFICIAL_REPO), '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--output-dir', str(OUTPUT_DIR), '--run-name', 'retro_scratch_colab',
        '--total-steps', str(PRETRAIN_STEPS), '--batch-size', '1', '--workers', '2',
        '--save-every', '1000', '--keep-last', '3', '--max-runtime-minutes', '300',
        '--baseline-milestone-complete',
    ])
else:
    print('Retrospective pretraining is OFF.')

## Export one selected training result

After training stops, select one checkpoint under this runtime's training-runs tree. This opt-in cell verifies the checkpoint, its `provenance.json` and `train.jsonl`, and compact JSON reports, then remounts Drive, copies them into a new export folder, verifies size/SHA-256, and unmounts Drive even if copying fails. It exports no raw data, slice caches, prediction trees, or submission archive.


In [ ]:
EXPORT_TRAINING_RESULT = False  # @param {type:'boolean'}
SELECTED_CHECKPOINT_VALUE = ''  # @param {type:'string'}
EXPORT_NAME_VALUE = 'selected_training_result'  # @param {type:'string'}
assert_runtime_only()
if EXPORT_TRAINING_RESULT:
    if not SELECTED_CHECKPOINT_VALUE.strip():
        raise ValueError('Select one runtime checkpoint before exporting.')
    checkpoint = require_runtime_path(SELECTED_CHECKPOINT_VALUE)
    if OUTPUT_DIR.resolve() not in checkpoint.parents:
        raise ValueError('Select a checkpoint from this runtime training-runs tree.')
    if checkpoint.parent.name != 'weights' or checkpoint.suffix != '.pth':
        raise ValueError('Select a training checkpoint from its weights/ directory.')
    run_dir = checkpoint.parent.parent
    selected = [checkpoint, run_dir / 'provenance.json', run_dir / 'train.jsonl']
    for path in selected:
        require_runtime_path(path)
        if not path.is_file():
            raise FileNotFoundError(f'Training artifact missing: {path}')
    selected.extend(sorted(REPORT_DIR.glob('*.json')))
    for path in selected:
        require_runtime_path(path)
    if len({path.name for path in selected}) != len(selected):
        raise ValueError('Selected export files have duplicate names.')
    if not EXPORT_NAME_VALUE or Path(EXPORT_NAME_VALUE).name != EXPORT_NAME_VALUE or EXPORT_NAME_VALUE in ('.', '..'):
        raise ValueError('Use one folder name for the export.')

    drive.mount(str(DRIVE_MOUNT))
    try:
        export_root = DRIVE_EXPORT_ROOT.expanduser().resolve()
        if drive_root not in export_root.parents:
            raise ValueError('Export destination must be inside mounted Drive.')
        export_dir = DRIVE_EXPORT_ROOT / EXPORT_NAME_VALUE
        if export_dir.exists():
            raise FileExistsError(f'Choose a new export name; folder already exists: {export_dir}')
        export_dir.mkdir(parents=True)
        copied = [
            copy_verified_file(path, export_dir / path.name) for path in selected
        ]
        (export_dir / 'export_manifest.json').write_text(
            json.dumps({'checkpoint': checkpoint.name, 'files': copied}, indent=2) + '\n',
            encoding='utf-8',
        )
    finally:
        drive.flush_and_unmount()
    assert_runtime_only()
    print('Verified training export:', export_dir)
else:
    print('Training export is OFF; results remain in runtime storage.')
